In [ ]:

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import os

# 1. 데이터 로드 및 전처리
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
sample_submission = pd.read_csv('../data/sample_submission.csv')

train = train.drop_duplicates(subset=[col for col in train.columns if col != 'ID']).reset_index(drop=True)
train['mean_working'] = train['mean_working'].fillna(0)
test['mean_working'] = test['mean_working'].fillna(0)

for col in ['medical_history', 'family_medical_history']:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

train['edu_level'] = train['edu_level'].fillna('Unknown')
test['edu_level'] = test['edu_level'].fillna('Unknown')

# 2. 파생변수 19개
def add_features(df):
    data = df.copy()
    has_disease = (data['medical_history'] != 'None').astype(int)
    data['is_overworking'] = (data['mean_working'] >= 10).astype(int)
    data['work_sleep_risk'] = ((data['mean_working'] >= 9) & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['oversleep_low_activity'] = ((data['sleep_pattern'] == 'oversleeping') & (data['activity'] == 'light')).astype(int)
    data['working_age_ratio'] = data['mean_working'] / (data['age'] + 1)
    data['activity_sleep_mismatch'] = ((data['activity'] == 'intense') & (data['sleep_pattern'] == 'sleep difficulty')).astype(int)
    data['smoker_with_disease'] = ((data['smoke_status'] == 'current-smoker') & (has_disease == 1)).astype(int)
    data['age_disease_interaction'] = data['age'] * has_disease
    data['has_medical_history'] = has_disease
    data['has_family_history'] = (data['family_medical_history'] != 'None').astype(int)
    data['total_disease_burden'] = data['has_medical_history'] + data['has_family_history']
    data['genetic_risk_match'] = ((data['medical_history'] == data['family_medical_history']) & (has_disease == 1)).astype(int)
    data['bmi'] = data['weight'] / ((data['height'] / 100) ** 2)
    data['pulse_pressure'] = data['systolic_blood_pressure'] - data['diastolic_blood_pressure']
    data['map'] = data['diastolic_blood_pressure'] + (data['pulse_pressure'] / 3)
    data['is_hypertension'] = ((data['systolic_blood_pressure'] >= 140) | (data['diastolic_blood_pressure'] >= 90)).astype(int)
    data['is_low_bone_density'] = (data['bone_density'] < 0).astype(int)
    data['glucose_chol_ratio'] = data['glucose'] / (data['cholesterol'] + 1)
    data['anticipatory_stress'] = ((data['family_medical_history'] != 'None') & (data['medical_history'] == 'None')).astype(int)
    data['cardio_metabolic_load'] = data['map'] * data['bmi']
    return data

train = add_features(train)
test = add_features(test)

# 3. 인코딩
activity_map = {'light': 0, 'moderate': 1, 'intense': 2}
edu_map = {'Unknown': 0, 'high school diploma': 1, 'bachelors degree': 2, 'graduate degree': 3}
train['activity'] = train['activity'].map(activity_map)
test['activity'] = test['activity'].map(activity_map)
train['edu_level'] = train['edu_level'].map(edu_map)
test['edu_level'] = test['edu_level'].map(edu_map)

nominal_cols = ['gender', 'smoke_status', 'medical_history', 'family_medical_history', 'sleep_pattern']
for feature in nominal_cols:
    le = LabelEncoder()
    le = le.fit(train[feature])
    train[feature] = le.transform(train[feature])
    unseen = [label for label in np.unique(test[feature]) if label not in le.classes_]
    if unseen: le.classes_ = np.append(le.classes_, unseen)
    test[feature] = le.transform(test[feature])

x_train = train.drop(['ID', 'stress_score'], axis=1)
y_train = train['stress_score']
x_test = test.drop('ID', axis=1)

# 4. 순수 LightGBM 5-Seed Ensemble
seeds = [42, 43, 44, 45, 46]
final_oof_preds = np.zeros(len(x_train))
final_test_preds = np.zeros(len(x_test))

print("=== 순수 LightGBM 시드 앙상블 학습 시작 ===")

for seed in seeds:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(x_train))
    seed_preds = np.zeros(len(x_test))
    
    # 06번 최고 성능 파라미터 유지
    model = LGBMRegressor(n_estimators=5000, learning_rate=0.03, num_leaves=127, min_child_samples=10, random_state=seed, verbose=-1)
    
    for tr_idx, va_idx in kf.split(x_train):
        X_tr, X_va = x_train.iloc[tr_idx], x_train.iloc[va_idx]
        y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
        
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
        
        seed_oof[va_idx] = model.predict(X_va)
        seed_preds += model.predict(x_test) / kf.n_splits
        
    final_oof_preds += seed_oof / len(seeds)
    final_test_preds += seed_preds / len(seeds)
    print(f"Seed {seed} 완료")

final_cv_mae = mean_absolute_error(y_train, final_oof_preds)
print(f"\n★ 최종 순수 LightGBM 앙상블 CV MAE: {final_cv_mae:.4f}")

# 5. 제출 파일 저장
os.makedirs('../submissions', exist_ok=True)
final_test_preds = np.clip(final_test_preds, 0, 1)
sample_submission['stress_score'] = final_test_preds
submit_path = '../submissions/submit_10_lgbm_seed_ensemble.csv'
sample_submission.to_csv(submit_path, index=False)
print(f"★ 제출 파일 생성 완료: {submit_path}")